In [1]:
!pip install -q transformers torch tqdm newspaper3k beautifulsoup4 requests


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 76.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 5.7 MB/s eta 0:00:00


In [4]:
pip install lxml[html_clean]

In [5]:
from transformers import pipeline
from tqdm import tqdm
import torch
import re
import requests
from bs4 import BeautifulSoup
from newspaper import Article


In [6]:
def clean_text(text):
    return re.sub(r'\s+', ' ', text.strip())


In [7]:
def chunk_text(text, max_chars=1000):

    chunks = []
    while len(text) > max_chars:
        split_at = text.rfind('.', 0, max_chars)
        if split_at == -1:
            split_at = max_chars
        chunks.append(text[:split_at+1])
        text = text[split_at+1:]
    if text:
        chunks.append(text)
    return chunks

In [8]:
def simple_sent_tokenize(text):
    return [s.strip() for s in re.split(r'(?<=[.!?]) +', text) if s]

In [11]:
def to_three_lines(summary_text):

    summary_text = clean_text(summary_text)
    sents = simple_sent_tokenize(summary_text)
    if len(sents) >= 3:
        return "\n".join(sents[:3])
    words = summary_text.split()
    n = len(words)
    sizes = [n // 3 + (1 if i < n % 3 else 0) for i in range(3)]
    lines, idx = [], 0
    for sz in sizes:
        lines.append(' '.join(words[idx:idx+sz]))
        idx += sz
    return "\n".join(lines)


In [12]:
hf_summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [13]:
def summarize_with_hf(text):
    text = clean_text(text)
    if len(text) < 50:
        return " Text too short to summarize."

    chunks = chunk_text(text, max_chars=1000)
    summaries = []

    for chunk in tqdm(chunks, desc="Summarizing"):
        if len(chunk.strip()) < 50:
            continue
        try:
            result = hf_summarizer(chunk, max_length=200, min_length=50, do_sample=False)
            summaries.append(result[0]['summary_text'])
        except Exception as e:
            print(f"Skipped chunk due to error: {e}")

    if not summaries:
        return "Could not generate summary. Text may be too short or fragmented."

    return to_three_lines(" ".join(summaries))



In [14]:
def fetch_article_text(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    paragraphs = [p.get_text() for p in soup.find_all('p')]
    text = ' '.join(paragraphs) if paragraphs else soup.get_text()
    title = soup.title.string if soup.title else "Untitled"
    return title, text



In [15]:
choice = input("Do you want to enter a URL or paste text? (url/text): ").strip().lower()

if choice == "url":
    url = input("Enter article URL: ").strip()
    try:
        title, text = fetch_article_text(url)
    except:
        print("Failed to fetch URL. Please paste text manually.")
        title = "Manual Input"
        text = input("Paste article text here: ")
elif choice == "text":
    text = input("Paste article text here: ")
    title = "Manual Input"
else:
    raise ValueError("Invalid choice. Enter 'url' or 'text'.")


Do you want to enter a URL or paste text? (url/text): url
Enter article URL: https://indianexpress.com/article/india/cyclone-shakhti-live-tracker-top-news-latest-updates-severe-cyclonic-storm-10287893/?ref=newlist_hp


In [16]:
three_line_summary = summarize_with_hf(text)


print(f"\n📰 Title: {title}\n")
print("3-Line Summary:\n", three_line_summary)

Summarizing: 100%|██████████| 6/6 [01:35<00:00, 15.89s/it]


📰 Title: Cyclone Shakhti Live Tracker News: Cyclone Shakti Alert IMD Warns of Heavy Rain in Mumbai, Pune, Thane and Other Maharashtra Districts Latest Update

3-Line Summary:
 The India Meteorological Department warned of thunderstorm and lightning as Cyclone Shakhti intensified into a ‘severe’ storm.
It was reportedly moving west-southwestwards of the Arabian Sea, clocking a speed of 13 km per hour during the last six hours.
The IMD has forecast rough sea conditions, squally weather, and high wind speeds ranging between 45-55 kmph gusting to 65 kmph at the Gujarat-Mahar Maharashtra coast till Tuesday.
